# GPT-2 Predictive Keyboard Contest Submission

This notebook runs the GPT-2 baseline model for the predictive keyboard contest. It uses the `transformers` library to predict the next word given a context and a first letter constraint.

## 1. Install Dependencies

In [ ]:
!pip install transformers torch pandas tqdm

## 2. Mount Google Drive

Mount your Google Drive to access the dataset files (`test_set_no_answer.csv`) and save the submission file.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set your data path here
DATA_DIR = '/content/drive/MyDrive/NLP/Contest2/' # CHANGE THIS TO YOUR PATH
TEST_FILE = DATA_DIR + 'test_set_no_answer.csv'
OUTPUT_FILE = DATA_DIR + 'submission_gpt2.txt'

## 3. Import Libraries & Define Predictor

In [ ]:
import torch
import pandas as pd
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from tqdm import tqdm
import logging
import time

# Suppress huggingface warnings
logging.getLogger("transformers").setLevel(logging.ERROR)

class GPT2Predictor:
    def __init__(self, model_name='gpt2', device=None):
        print(f"Loading {model_name}...")
        self.device = device if device else ('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")
        
        self.tokenizer = GPT2Tokenizer.from_pretrained(model_name)
        # GPT-2 has no pad token by default, set to EOS
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        self.model = GPT2LMHeadModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def predict_batch(self, contexts, first_letters, batch_size=32):
        """
        Predicts next words for a batch. Optimized for GPU.
        """
        all_predictions = []
        
        # Process in batches
        for i in range(0, len(contexts), batch_size):
            batch_contexts = contexts[i : i + batch_size]
            batch_first_letters = first_letters[i : i + batch_size]
            
            # Tokenize
            inputs = self.tokenizer(batch_contexts, return_tensors='pt', padding=True, truncation=True, max_length=1024).to(self.device)
            
            with torch.no_grad():
                outputs = self.model(**inputs)
                logits = outputs.logits

            # Iterate through batch items
            for j, (context, first_letter) in enumerate(zip(batch_contexts, batch_first_letters)):
                # Identify the last real token (ignoring padding)
                # attention_mask is 1 for real tokens, 0 for padding
                seq_len = inputs.attention_mask[j].sum().item()
                
                # Logits for the next word are at the last position of the input sequence
                # (index = seq_len - 1)
                next_token_logits = logits[j, seq_len - 1, :]
                
                # Get top-k candidates
                k = 1000
                top_k_indices = torch.topk(next_token_logits, k).indices.tolist()
                
                found_word = ""
                for idx in top_k_indices:
                    # Decode token
                    word = self.tokenizer.decode([idx]).strip().lower()
                    
                    # Check if it matches
                    # 1. Must start with first_letter
                    if len(word) > 0 and word.startswith(first_letter.lower()):
                        # 2. Must not be a special token
                        if word not in ["<|endoftext|>", "<pad>"]:
                            found_word = word
                            break
                
                # Fallback if nothing found
                if not found_word:
                     # Fallback to just the letter itself if possible or empty
                     found_word = first_letter
                     
                all_predictions.append(found_word)
                
        return all_predictions

In [ ]:
# Set to True if you want to run evaluation
RUN_EVALUATION = True
DEV_FILE = DATA_DIR + 'dev_set.csv'

if RUN_EVALUATION:
    print(f"Loading dev data from {DEV_FILE}...")
    try:
        dev_df = pd.read_csv(DEV_FILE)
        # Limit for speed if needed, or remove for full eval
        # dev_df = dev_df.head(1000) 
        
        print(f"Evaluating on {len(dev_df)} examples...")
        
        contexts = dev_df['context'].tolist()
        first_letters = dev_df['first letter'].tolist()
        answers = dev_df['answer'].tolist()
        
        # Batch Prediction
        EVAL_BATCH_SIZE = 64
        predictions = []
        
        predictor = GPT2Predictor(model_name='gpt2', device='cuda')
        
        for i in tqdm(range(0, len(contexts), EVAL_BATCH_SIZE)):
            batch_contexts = contexts[i : i + EVAL_BATCH_SIZE]
            batch_first_letters = first_letters[i : i + EVAL_BATCH_SIZE]
            
            batch_preds = predictor.predict_batch(batch_contexts, batch_first_letters, batch_size=EVAL_BATCH_SIZE)
            predictions.extend(batch_preds)
            
        # Calculate Accuracy
        correct = 0
        for pred, true_ans in zip(predictions, answers):
            if pred == true_ans:
                correct += 1
                
        accuracy = correct / len(predictions)
        print(f"Accuracy: {accuracy:.4f}")
        
    except FileNotFoundError:
        print(f"Dev file not found at {DEV_FILE}. Skipping evaluation.")

## 4. Evaluate on Dev Set
Optionally, load the development set to check accuracy before generating the submission.

## 4. Run Inference

In [ ]:
# Initialize Predictor
predictor = GPT2Predictor(model_name='gpt2', device='cuda')

# Load Test Data
print(f"Loading test data from {TEST_FILE}...")
try:
    test_df = pd.read_csv(TEST_FILE)
except FileNotFoundError:
    print("File not found. Making a dummy dataframe for demonstration.")
    test_df = pd.DataFrame({'context': ['This is a test', 'Hello world'], 'first letter': ['c', 'w']})

contexts = test_df['context'].tolist()
first_letters = test_df['first letter'].tolist()

print(f"Generating predictions for {len(contexts)} examples...")

# Batch Prediction
BATCH_SIZE = 64 # Adjust based on GPU memory. Colab T4 can handle 32-64 well.
predictions = []

# Processing in chunks manually to show progress bar
for i in tqdm(range(0, len(contexts), BATCH_SIZE)):
    batch_contexts = contexts[i : i + BATCH_SIZE]
    batch_first_letters = first_letters[i : i + BATCH_SIZE]
    
    batch_preds = predictor.predict_batch(batch_contexts, batch_first_letters, batch_size=BATCH_SIZE)
    predictions.extend(batch_preds)

print("Done!")

## 5. Save Submission

In [ ]:
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for pred in predictions:
        f.write(f"{pred}\n")
        
print(f"Saved submission to {OUTPUT_FILE}")